In [3]:
from unike.module.model import RGCN, CompGCN
import sys

sys.path.extend(['.', '..'])

from q import link, drug_ent_indexs, add_id

In [4]:
RGCN_model = RGCN(
	ent_tol = 121649,
	rel_tol = 22,
	dim = 200,
	num_layers = 2
)
RGCN_model.load_checkpoint("/home/wangtao/src/kg4rd/src/kg4rd/kge/checkpoints/A/RGCN_entrie_Accel_20250910-1000.pth")

CompGCN_model = CompGCN(
    ent_tol = 121649,
    rel_tol = 22,
    dim = 50
)
CompGCN_model.load_checkpoint("/home/wangtao/src/kg4rd/src/kg4rd/kge/checkpoints/A/CompGCN_entrie_Accel_20250910-1000.pth")

In [31]:
RGCN_result = add_id(link.link(
    drug_ent_indexs,
    [1],
    [9315, 4289, 13239, 7993, 9016, 7509], # UTRN, COL6A3, DYSF, DOK7, DMD, SGCD
    RGCN_model, 'cuda:0'
))

In [32]:
CompGCN_result = add_id(link.link(
    drug_ent_indexs,
    [1],
    [9315, 4289, 13239, 7993, 9016, 7509], # UTRN, COL6A3, DYSF, DOK7, DMD, SGCD
    CompGCN_model, 'cuda:0'
))

In [33]:
topk = 300
RGCN_pairs = set(zip(RGCN_result['head'].head(topk), RGCN_result['tail'].head(topk)))
CompGCN_pairs = set(zip(CompGCN_result['head'].head(topk), CompGCN_result['tail'].head(topk)))
intersect_pairs = RGCN_pairs & CompGCN_pairs
mask = list(zip(RGCN_result['head'], RGCN_result['tail']))
RGCN_result[[p in intersect_pairs for p in mask]].to_csv(f"./target_intersect_result_{topk}.csv", index=False)